In [0]:
# ============================================================
# BANKING DATA ENGINEERING PROJECT
# GOLD LAYER - DAILY BANKING KPIs
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    max,
    to_date
)


# ============================================================
# SPARK SESSION
# ============================================================

spark = SparkSession.builder.appName(
    "Banking_Gold_Daily_KPIs"
).getOrCreate()


# ============================================================
# CONFIGURATION
# ============================================================

silver_path = (
    "/Volumes/bank_data/default/bank_data/"
    "silver_bank_transactions"
)

gold_path = (
    "/Volumes/bank_data/default/bank_data/"
    "gold_daily_banking_kpis"
)


# ============================================================
# READ SILVER
# ============================================================

print("========================================")
print("            READ SILVER")
print("========================================")


silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)


silver_count = silver_df.count()

print(f"Silver Rows: {silver_count}")


# ============================================================
# GOLD TRANSFORMATION
# ============================================================

print("\n========================================")
print("        GOLD TRANSFORMATION")
print("========================================")


gold_df = (
    silver_df

    .withColumn(
        "Transaction_Date",
        to_date(col("DATE"))
    )

    .groupBy("Transaction_Date")

    .agg(

        count("*").alias(
            "Transaction_Count"
        ),

        sum(
            col("WITHDRAWAL_AMT")
        ).alias(
            "Total_Withdrawal"
        ),

        sum(
            col("DEPOSIT_AMT")
        ).alias(
            "Total_Deposit"
        ),

        avg(
            col("WITHDRAWAL_AMT")
        ).alias(
            "Avg_Withdrawal"
        ),

        avg(
            col("DEPOSIT_AMT")
        ).alias(
            "Avg_Deposit"
        ),

        max(
            col("BALANCE_AMT")
        ).alias(
            "Max_Balance"
        )
    )
)


# ============================================================
# WRITE GOLD
# ============================================================

print("\n========================================")
print("            WRITE GOLD")
print("========================================")


(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)


print(
    "Gold Daily Banking KPIs written successfully"
)


# ============================================================
# READ GOLD FOR VALIDATION
# ============================================================

gold_check_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)


gold_count = gold_check_df.count()

gold_columns = len(
    gold_check_df.columns
)


# ============================================================
# GOLD VALIDATION
# ============================================================

print("\n========================================")
print("          GOLD VALIDATION")
print("========================================")


print(
    f"Silver Rows: {silver_count}"
)

print(
    f"Gold Daily Rows: {gold_count}"
)

print(
    f"Gold Columns: {gold_columns}"
)


# ============================================================
# EXPECTED RESULTS
# ============================================================

expected_days = 1294

expected_columns = 7


if gold_count == expected_days:

    print(
        " DAILY ROW COUNT VALIDATION PASSED"
    )

else:

    print(
        " DAILY ROW COUNT VALIDATION FAILED"
    )


if gold_columns == expected_columns:

    print(
        " COLUMN COUNT VALIDATION PASSED"
    )

else:

    print(
        " COLUMN COUNT VALIDATION FAILED"
    )


# ============================================================
# NULL VALIDATION
# ============================================================

print("\n========================================")
print("           NULL VALIDATION")
print("========================================")


null_dates = (
    gold_check_df
    .filter(
        col("Transaction_Date").isNull()
    )
    .count()
)


null_transaction_count = (
    gold_check_df
    .filter(
        col("Transaction_Count").isNull()
    )
    .count()
)


print(
    f"NULL Transaction_Date: {null_dates}"
)

print(
    f"NULL Transaction_Count: {null_transaction_count}"
)


if null_dates == 0:

    print(
        " Transaction_Date NULL CHECK PASSED"
    )

else:

    print(
        " Transaction_Date NULL CHECK FAILED"
    )


if null_transaction_count == 0:

    print(
        "Transaction_Count NULL CHECK PASSED"
    )

else:

    print(
        "Transaction_Count NULL CHECK FAILED"
    )


# ============================================================
# BUSINESS VALIDATION
# ============================================================

print("\n========================================")
print("       BUSINESS VALIDATION")
print("========================================")


invalid_transaction_days = (
    gold_check_df
    .filter(
        col("Transaction_Count") <= 0
    )
    .count()
)


print(
    f"Days with invalid transaction count: "
    f"{invalid_transaction_days}"
)


if invalid_transaction_days == 0:

    print(
        "TRANSACTION COUNT BUSINESS CHECK PASSED"
    )

else:

    print(
        "TRANSACTION COUNT BUSINESS CHECK FAILED"
    )


# ============================================================
# SCHEMA
# ============================================================

print("\n========================================")
print("             GOLD SCHEMA")
print("========================================")


gold_check_df.printSchema()


# ============================================================
# SAMPLE DATA
# ============================================================

print("\n========================================")
print("          GOLD SAMPLE DATA")
print("========================================")


display(
    gold_check_df.orderBy(
        col("Transaction_Date")
    )
)


# ============================================================
# FINAL DATA QUALITY CHECK
# ============================================================

if (
    gold_count == expected_days
    and gold_columns == expected_columns
    and null_dates == 0
    and null_transaction_count == 0
    and invalid_transaction_days == 0
):

    print("\n========================================")
    print("    GOLD DAILY KPIs COMPLETED")
    print("========================================")

    print(
        " ALL GOLD DAILY KPI CHECKS PASSED"
    )

    print(
        f"Days: {gold_count}"
    )

    print(
        f"Columns: {gold_columns}"
    )

    print(
        f"Gold Delta Path: {gold_path}"
    )

else:

    raise Exception(
        "GOLD DAILY KPI DATA QUALITY CHECK FAILED"
    )

            READ SILVER
Silver Rows: 116162

        GOLD TRANSFORMATION

            WRITE GOLD
Gold Daily Banking KPIs written successfully

          GOLD VALIDATION
Silver Rows: 116162
Gold Daily Rows: 1294
Gold Columns: 7
 DAILY ROW COUNT VALIDATION PASSED
 COLUMN COUNT VALIDATION PASSED

           NULL VALIDATION
NULL Transaction_Date: 0
NULL Transaction_Count: 0
 Transaction_Date NULL CHECK PASSED
Transaction_Count NULL CHECK PASSED

       BUSINESS VALIDATION
Days with invalid transaction count: 0
TRANSACTION COUNT BUSINESS CHECK PASSED

             GOLD SCHEMA
root
 |-- Transaction_Date: date (nullable = true)
 |-- Transaction_Count: long (nullable = true)
 |-- Total_Withdrawal: double (nullable = true)
 |-- Total_Deposit: double (nullable = true)
 |-- Avg_Withdrawal: double (nullable = true)
 |-- Avg_Deposit: double (nullable = true)
 |-- Max_Balance: double (nullable = true)


          GOLD SAMPLE DATA


Transaction_Date,Transaction_Count,Total_Withdrawal,Total_Deposit,Avg_Withdrawal,Avg_Deposit,Max_Balance
2015-01-01,3,null,2004999.0,null,668333.0,-5.399581216100013E8
2015-01-02,10,5500056.18,2465620.0,2750028.09,308202.5,-5.399551216100013E8
2015-01-03,4,2200000.0,2948160.0,2200000.0,982720.0,-1.5872048654300041E9
2015-01-05,12,3231000.0,3909020.0,1615500.0,390902.0,-5.399234216100013E8
2015-01-06,6,3880056.18,3650750.0,1940028.09,912687.5,-1.5865586016100042E9
2015-01-07,4,3650000.0,3455070.0,3650000.0,1151690.0,-1.5869827816100042E9
2015-01-08,4,3450000.0,4550200.0,3450000.0,1516733.3333333333,-1.5858825816100042E9
2015-01-09,4,4550000.0,4902100.0,4550000.0,1634033.3333333333,-1.5855304816100042E9
2015-01-10,12,3901500.0,5614880.0,1300500.0,623875.5555555555,-1.5838171016100042E9
2015-01-12,4,6600000.0,6957710.0,6600000.0,2319236.6666666665,-1.5834593916100042E9



    GOLD DAILY KPIs COMPLETED
 ALL GOLD DAILY KPI CHECKS PASSED
Days: 1294
Columns: 7
Gold Delta Path: /Volumes/bank_data/default/bank_data/gold_daily_banking_kpis
